# Derivatives Matching - Complete BRD Rules + Greedy Layer

Document Reference: Derivatives.docx (Business Requirements Document)

Purpose:
Implement ALL matching rules from the BRD + Greedy/Probabilistic Layer

Layer 1: BRD Deterministic Rules (15 total)
- 3 SOPHIS-specific rules
- 10 OTC rules
- 2 ETD rules

Layer 2: Greedy/Probabilistic Matching
- Strategy 1: Amount + Counterparty (1% tolerance)
- Strategy 2: Amount-only strict (0.1% tolerance)

Date: 2026-01-28 (v3 - BRD + Greedy)

Changes from Previous Version (v2):
- Added configurable SOPHIS/DELTA1 scope exclusion
- Added Greedy matching layer for remaining unmatched trades
- Expected match rate: ~87% (with SOPHIS/DELTA1 excluded)

SCOPE NOTE:
SOPHIS and DELTA1 systems are excluded by default.
Their trade IDs do not exist in Finstore – this is a data mapping issue, not a matching algorithm issue.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, List, Dict, Optional, Set
from dataclasses import dataclass, field
from enum import Enum
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

# ============================================================
# SCOPE CONFIGURATION
# ============================================================

# Set to True to exclude SOPHIS and DELTA1 systems from scope
# These systems' trade IDs don't exist in Finstore (data mapping issue)
EXCLUDE_SOPHIS_DELTA1 = True

# Out-of-scope systems (when EXCLUDE_SOPHIS_DELTA1 = True)
OUT_OF_SCOPE_SYSTEMS = [
    "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO", "SOPHISFX-LONDON",
    "DELTA1-LONDON", "DELTA1-NEWYORK"
]

# Greedy matching parameters
GREEDY_TOLERANCE_PCT = 0.01    # 1% for Strategy 1 (Amount + Counterparty)
STRICT_TOLERANCE_PCT = 0.001   # 0.1% for Strategy 2 (Amount only)


# ============================================================
# FILE PATHS
# ============================================================

WORKING_DIR = Path.cwd()

INPUT_FILE_FINSTORE = WORKING_DIR / "finstore_sample_poc.csv"
INPUT_FILE_AXIS = WORKING_DIR / "axis_sample_poc.csv"

# Optional: SDS mapping file for DerivedMasterbookId (if available)
SDS_MAPPING_FILE = WORKING_DIR / "sds_book_mapping.csv"

# Output files
OUTPUT_DIR = WORKING_DIR / "greedy_results"
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_MATCHED_BRD = OUTPUT_DIR / "matched_brd_layer.csv"
OUTPUT_MATCHED_GREEDY = OUTPUT_DIR / "matched_greedy_layer.csv"
OUTPUT_MATCHED_ALL = OUTPUT_DIR / "matched_all_combined.csv"
OUTPUT_UNMATCHED_AXIS = OUTPUT_DIR / "unmatched_axis_final.csv"
OUTPUT_UNMATCHED_FINSTORE = OUTPUT_DIR / "unmatched_finstore_final.csv"
OUTPUT_SUMMARY = OUTPUT_DIR / "matching_summary_report.txt"


print(f"Working Directory: {WORKING_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print("")

print("SCOPE CONFIGURATION:")
print(f"  EXCLUDE_SOPHIS_DELTA1: {EXCLUDE_SOPHIS_DELTA1}")

if EXCLUDE_SOPHIS_DELTA1:
    print(f"  Excluded systems: {', '.join(OUT_OF_SCOPE_SYSTEMS)}")

print("")
print("GREEDY MATCHING PARAMETERS:")
print(f"  Strategy 1 tolerance: {GREEDY_TOLERANCE_PCT * 100}%")
print(f"  Strategy 2 tolerance: {STRICT_TOLERANCE_PCT * 100}%")

Working Directory: C:\Users\anandv\OneDrive - Barclays Bank PLC
Output Directory: C:\Users\anandv\OneDrive - Barclays Bank PLC\greedy_results

SCOPE CONFIGURATION:
  EXCLUDE_SOPHIS_DELTA1: True
  Excluded systems: SOPHIS-LONDON, SOPHIS-NEWYORK, SOPHIS-TOKYO, SOPHISFX-LONDON, DELTA1-LONDON, DELTA1-NEWYORK

GREEDY MATCHING PARAMETERS:
  Strategy 1 tolerance: 1.0%
  Strategy 2 tolerance: 0.1%

🔹 Section 2: BRD Constants and Classifications


In [ ]:
# ============================================================
# BRD SECTION: Recon SubProduct Classification
# ============================================================

class ReconSubProduct(Enum):
    OTC = "OTC"
    ETD = "ETD"
    OTC_DEFAULT = "OTC-Default"
    
    
# OTC Systems (from BRD table)
OTC_SYSTEMS: Set[str] = {
    "ALD-SF", "BARXFX-PD-ASIAPAC", "BARXFX-PD-LONDON", "BARXFETS",
    "BARXFX-TS-CASH-ASIAPAC", "BARXFX-TS-CASH-LONDON", "DELTA1-LONDON",
    "DELTA1-NEWYORK", "IFLOW-AMERICAS", "IFLOW-ASIAPACIFIC", "IFLOW-EUROPE",
    "FISSFX-NEWYORK", "FISS-LONDON", "FISS-TOKYO", "GCD-NEWYORK",
    "IMPACT-NEWYORK", "OPENLINK-LONDON", "OPTISRD-MEXICO", "SABS-NEWYORK",
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO",
    "SUMMIT-LONDON", "SYNFINY-CFD-NONCASH", "SYETR-NEWYORK-MBS"
}

# ETD Systems (from BRD table)
ETD_SYSTEMS: Set[str] = {
    "BPS222-OPT-NEWYORK", "BPS224-OPT-NEWYORK", "ODH-DOLPHIN-INDIA",
    "ODH-CMI-CPM-LONDON", "ODH-GMI-HONGKONG", "ODH-GMI-LONDON",
    "ODH-GMI-LONDON-BDT", "ODH-GMI-NEWYORK", "ODH-GMI-SINGAPORE",
    "ODH-ISTAR-TOKYO", "OPTISFUT-MEXICO"
}

# SOPHIS Systems
SOPHIS_SYSTEMS: Set[str] = {
    "SOPHISFX-LONDON",
    "SOPHIS-LONDON",
    "SOPHIS-TOKYO",
    "SOPHIS-NEWYORK"
}

# DELTA1 Systems
DELTA1_SYSTEMS: Set[str] = {
    "DELTA1-LONDON",
    "DELTA1-NEWYORK"
}
# Amount columns
AXIS_AMOUNT_COLUMN = "SACCRMTM"
FINSTORE_AMOUNT_COLUMN = "gbpequivalentamount"


print("BRD CONSTANTS LOADED")
print(f"OTC Systems: {len(OTC_SYSTEMS)}")
print(f"ETD Systems: {len(ETD_SYSTEMS)}")
print(f"SOPHIS Systems: {len(SOPHIS_SYSTEMS)}")
print(f"DELTA1 Systems: {len(DELTA1_SYSTEMS)}")

🔹 Section 3: Matching Rule Definitions (All 15 BRD Rules)


In [ ]:
class MatchingCategory(Enum):
    SOPHIS = "SOPHIS"
    OTC = "OTC"
    ETD = "ETD"
    
@dataclass
class MatchingRule:
    category: MatchingCategory
    brd_priority: int
    description: str
    axis_columns: List[str]
    finstore_columns: List[str]
    requires_derived_masterbook: bool = False
    requires_derived_sophis: bool = False
    requires_derived_delta1: bool = False
    
# SOPHIS specific Rules (3)
SOPHIS_MATCHING_RULES: List[MatchingRule] = [

    MatchingRule(
        MatchingCategory.SOPHIS, 1,
        "ExtractedSophisId <> FissNumber",
        ["DerivedSophisId"],
        ["FissNumber"],
        requires_derived_sophis=True
    ),

    MatchingRule(
        MatchingCategory.SOPHIS, 2,
        "ExtractedSophisId + FissNumber + TradingSystem",
        ["DerivedSophisId", "BookId"],
        ["FissNumber", "TradingSystem"],
        requires_derived_sophis=True
    ),

    MatchingRule(
        MatchingCategory.SOPHIS, 3,
        "ExtractedSophisId <> TradeId",
        ["DerivedSophisId"],
        ["TradeId"],
        requires_derived_sophis=True
    )
]

# OTC Rules (10)
OTC_MATCHING_RULES: List[MatchingRule] = [

    MatchingRule(
        MatchingCategory.OTC, 1,
        "SourceSystemTradeId <> TradeId",
        ["SourceSystemTradeId"],
        ["TradeId"]
    ),

    MatchingRule(
        MatchingCategory.OTC, 2,
        "SourceSystemTradeId + DerivedMasterbookId <> TradeId + MasterbookId",
        ["SourceSystemTradeId", "DerivedMasterbookId"],
        ["TradeId", "MasterbookId"],
        requires_derived_masterbook=True
    ),

    MatchingRule(
        MatchingCategory.OTC, 3,
        "SourceSystemTradeId <> AlternativeTradeId",
        ["SourceSystemTradeId"],
        ["AlternativeTradeId"]
    ),

    MatchingRule(
        MatchingCategory.OTC, 4,
        "SourceSystemTradeId + DerivedMasterbookId <> AlternativeTradeId + MasterbookId",
        ["SourceSystemTradeId", "DerivedMasterbookId"],
        ["AlternativeTradeId", "MasterbookId"],
        requires_derived_masterbook=True
    ),

    MatchingRule(
        MatchingCategory.OTC, 5,
        "DerivedSophisId <> FissNumber",
        ["DerivedSophisId"],
        ["FissNumber"],
        requires_derived_sophis=True
    ),

    MatchingRule(
        MatchingCategory.OTC, 6,
        "DerivedSophisId + BookId <> FissNumber + TradingSystem",
        ["DerivedSophisId", "BookId"],
        ["FissNumber", "TradingSystem"],
        requires_derived_sophis=True
    ),

    MatchingRule(
        MatchingCategory.OTC, 7,
        "ExtractedSophisId <> TradeId",
        ["DerivedSophisId"],
        ["TradeId"]
    ),

    MatchingRule(
        MatchingCategory.OTC, 8,
        "ExtractedDelta1Id <> TradeId",
        ["DerivedDelta1Id"],
        ["TradeId"],
        requires_derived_delta1=True
    ),

    MatchingRule(
        MatchingCategory.OTC, 9,
        "SourceSystemTradeId <> AlternativeTradeId",
        ["SourceSystemTradeId"],
        ["AlternativeTradeId"]
    ),

    MatchingRule(
        MatchingCategory.OTC, 10,
        "SourceSystemTradeId + DerivedMasterbookId <> AlternativeTradeId + MasterbookId",
        ["SourceSystemTradeId", "DerivedMasterbookId"],
        ["AlternativeTradeId", "MasterbookId"],
        requires_derived_masterbook=True
    )
]

# ETD Rules (2)
ETD_MATCHING_RULES: List[MatchingRule] = [

    MatchingRule(
        MatchingCategory.ETD, 1,
        "SourceSystemInstrumentId + DerivedMasterbookId <> InstrumentId + MasterbookId",
        ["SourceSystemInstrumentId", "DerivedMasterbookId"],
        ["InstrumentId", "MasterbookId"],
        requires_derived_masterbook=True
    ),

    MatchingRule(
        MatchingCategory.ETD, 2,
        "SourceSystemInstrumentId <> InstrumentId",
        ["SourceSystemInstrumentId"],
        ["InstrumentId"]
    )
]

ALL_MATCHING_RULES = (
    SOPHIS_MATCHING_RULES
    + OTC_MATCHING_RULES
    + ETD_MATCHING_RULES
)

print(f"SOPHIS-specific rules: {len(SOPHIS_MATCHING_RULES)}")
print(f"OTC rules: {len(OTC_MATCHING_RULES)}")
print(f"ETD rules: {len(ETD_MATCHING_RULES)}")
print(f"TOTAL: {len(ALL_MATCHING_RULES)} rules")

🔹 Section 4: Load Data


In [ ]:
# Load Axis and Finstore data
logger.info("Loading Axis data...")
df_axis_full = pd.read_csv(INPUT_FILE_AXIS, low_memory=False)
ORIGINAL_AXIS_COUNT_FULL = len(df_axis_full)
logger.info(f"Axis loaded: {ORIGINAL_AXIS_COUNT_FULL:,} records")

logger.info("Loading Finstore data...")
df_finstore = pd.read_csv(INPUT_FILE_FINSTORE, low_memory=False)
ORIGINAL_FINSTORE_COUNT = len(df_finstore)
logger.info(f"Finstore loaded: {ORIGINAL_FINSTORE_COUNT:,} records")


# Load SDS mapping if available
df_sds_mapping = None
if SDS_MAPPING_FILE.exists():
    logger.info("Loading SDS mapping data...")
    df_sds_mapping = pd.read_csv(SDS_MAPPING_FILE, low_memory=False)
    logger.info(f"SDS mapping loaded: {len(df_sds_mapping):,} records")
    
print(f"\n{'='*80}")
print("DATA LOADED")
print(f"{'='*80}")

print(f"Axis (full):   {ORIGINAL_AXIS_COUNT_FULL:,} records")
print(f"Finstore:      {ORIGINAL_FINSTORE_COUNT:,} records")
print(f"SDS:           {'Available' if df_sds_mapping is not None else 'NOT AVAILABLE'}")

🔹 Section 5: Apply Scope Exclusion (SOPHIS / DELTA1)


In [ ]:
print(f"\n{'='*80}")
print("SCOPE EXCLUSION")
print(f"{'='*80}")

if EXCLUDE_SOPHIS_DELTA1:

    # Count excluded records
    excluded_mask = df_axis_full["SourceSystemName"].isin(OUT_OF_SCOPE_SYSTEMS)
    df_axis_excluded = df_axis_full[excluded_mask].copy()
    df_axis = df_axis_full[~excluded_mask].copy()

    print("\n*** SOPHIS/DELTA1 SCOPE EXCLUSION ACTIVE ***")
    print("")
    print("Excluded systems breakdown:")

    excluded_by_system = df_axis_excluded["SourceSystemName"].value_counts()
    for sys, count in excluded_by_system.items():
        print(f"  {sys}: {count:,}")

    print("-" * 50)
    print(f"TOTAL EXCLUDED: {len(df_axis_excluded):,}")
    print("")

    print("Scope Summary:")
    print(f"  Original Axis: {ORIGINAL_AXIS_COUNT_FULL:,}")
    print(f"  Excluded:      {len(df_axis_excluded):,} ({(len(df_axis_excluded)/ORIGINAL_AXIS_COUNT_FULL*100):.1f}%)")
    print(f"  In-Scope:      {len(df_axis):,} ({(len(df_axis)/ORIGINAL_AXIS_COUNT_FULL*100):.1f}%)")

else:
    df_axis = df_axis_full.copy()
    df_axis_excluded = pd.DataFrame()
    print(f"\nAll systems in scope: {len(df_axis):,}")

ORIGINAL_AXIS_COUNT = len(df_axis)  # This is our working count

print(f"\nWorking Axis count (in-scope): {ORIGINAL_AXIS_COUNT:,}")
print(f"{'='*80}")

🔹 Section 6: Pre-Reconciliation Derivations


In [ ]:
def extract_third_part(trade_id: str) -> str:
    """Extract the 3rd part of a trade ID string split by '-'."""
    if pd.isna(trade_id):
        return ""
    parts = str(trade_id).split("-")
    if len(parts) >= 3:
        return parts[2]
    return ""

def classify_recon_subproduct(source_system: str) -> str:
    """Classify source system into Recon SubProduct."""
    if source_system in ETD_SYSTEMS:
        return "ETD"
    elif source_system in OTC_SYSTEMS:
        return "OTC"
    else:
        return "OTC-Default"
    
# Add tracking indices
df_axis["OriginalAxisIndex"] = df_axis.index
df_finstore["OriginalFinstoreIndex"] = df_finstore.index

# Derive ReconSubProduct
df_axis["ReconSubProduct"] = df_axis["SourceSystemName"].apply(classify_recon_subproduct)

# Derive DerivedSophisId
df_axis["DerivedSophisId"] = ""

sophis_mask = df_axis["SourceSystemName"].isin(SOPHIS_SYSTEMS)
if sophis_mask.sum() > 0:
    df_axis.loc[sophis_mask, "DerivedSophisId"] = (
        df_axis.loc[sophis_mask, "SourceSystemTradeId"]
        .apply(extract_third_part)
    )
    
# Derive DerivedSophisId
df_axis["DerivedSophisId"] = ""

sophis_mask = df_axis["SourceSystemName"].isin(SOPHIS_SYSTEMS)
if sophis_mask.sum() > 0:
    df_axis.loc[sophis_mask, "DerivedSophisId"] = (
        df_axis.loc[sophis_mask, "SourceSystemTradeId"]
        .apply(extract_third_part)
    )

# Derive DerivedDelta1Id
df_axis["DerivedDelta1Id"] = ""

delta1_mask = df_axis["SourceSystemName"].isin(DELTA1_SYSTEMS)
if delta1_mask.sum() > 0:
    df_axis.loc[delta1_mask, "DerivedDelta1Id"] = (
        df_axis.loc[delta1_mask, "SourceSystemTradeId"]
        .apply(extract_third_part)
    )
# DerivedMasterbookId placeholder (requires SDS)
df_axis["DerivedMasterbookId"] = ""

print("PRE-RECONCILIATION DERIVATIONS COMPLETE")
print("ReconSubProduct Distribution:")
print(df_axis["ReconSubProduct"].value_counts())
print("")
print("Derived Columns:")
print(f"  DerivedSophisId populated: {(df_axis['DerivedSophisId'] != '').sum():,}")
print(f"  DerivedDelta1Id populated: {(df_axis['DerivedDelta1Id'] != '').sum():,}")



PRE-RECONCILIATION DERIVATIONS COMPLETE
ReconSubProduct Distribution:
OTC    57361
ETD     8544
Name: ReconSubProduct, dtype: int64

🔹 Section 7: Matching Engine Core Functions


In [ ]:
def create_composite_key(
    df: pd.DataFrame,
    columns: List[str],
    key_name: str
) -> pd.Series:
    """Create a composite key from multiple columns."""
    if len(columns) == 1:
        return df[columns[0]].astype(str).str.strip()

    composite = df[columns[0]].astype(str).str.strip()
    for col in columns[1:]:
        composite = composite + "_" + df[col].astype(str).str.strip()

    return composite

def execute_matching_rule(
    df_axis_pool: pd.DataFrame,
    df_finstore_pool: pd.DataFrame,
    rule: MatchingRule,
    global_priority: int
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict]:
    """Execute a single matching rule."""

    rule_desc = f"{rule.category.value} Rule {rule.brd_priority}: {rule.description}"

    # If empty pools
    if len(df_axis_pool) == 0 or len(df_finstore_pool) == 0:
        return pd.DataFrame(), df_axis_pool, df_finstore_pool, {
            "global_priority": global_priority,
            "category": rule.category.value,
            "brd_priority": rule.brd_priority,
            "description": rule.description,
            "matched_count": 0,
            "matched_unique_axis": 0
        }
    # Check required derived columns
    if rule.requires_derived_masterbook:
        if (df_axis_pool["DerivedMasterbookId"] != "").sum() == 0:
            return pd.DataFrame(), df_axis_pool, df_finstore_pool, {
                "global_priority": global_priority,
                "category": rule.category.value,
                "brd_priority": rule.brd_priority,
                "description": rule.description,
                "matched_count": 0,
                "matched_unique_axis": 0,
                "skipped_reason": "DerivedMasterbookId not available"
            }
    # Create key names
    axis_key_col = f"_match_key_axis_{global_priority}"
    finstore_key_col = f"_match_key_finstore_{global_priority}"

    df_axis_work = df_axis_pool.copy()
    df_finstore_work = df_finstore_pool.copy()

    df_axis_work[axis_key_col] = create_composite_key(
        df_axis_work, rule.axis_columns, axis_key_col
    )

    df_finstore_work[finstore_key_col] = create_composite_key(
        df_finstore_work, rule.finstore_columns, "finstore_key"
    )
    # Filter valid keys
    axis_valid = df_axis_work[
        (df_axis_work[axis_key_col] != "") &
        (df_axis_work[axis_key_col] != "nan") &
        (~df_axis_work[axis_key_col].str.contains(r"\$", regex=True))
    ].copy()

    finstore_valid = df_finstore_work[
        (df_finstore_work[finstore_key_col] != "") &
        (df_finstore_work[finstore_key_col] != "nan") &
        (~df_finstore_work[finstore_key_col].str.contains(r"\$", regex=True))
    ].copy()
    if len(axis_valid) == 0 or len(finstore_valid) == 0:
        return pd.DataFrame(), df_axis_pool, df_finstore_pool, {
            "global_priority": global_priority,
            "category": rule.category.value,
            "brd_priority": rule.brd_priority,
            "description": rule.description,
            "matched_count": 0,
            "matched_unique_axis": 0
        }
        # Merge
    matched = pd.merge(
        axis_valid,
        finstore_valid,
        left_on=axis_key_col,
        right_on=finstore_key_col,
        how="inner",
        suffixes=("_Axis", "_Finstore")
    )
    if len(matched) > 0:

        matched["MatchGlobalPriority"] = global_priority
        matched["MatchCategory"] = rule.category.value
        matched["MatchBRDPriority"] = rule.brd_priority
        matched["MatchDescription"] = rule.description
        matched["MatchLayer"] = "BRD"
        # Calculate variance
        axis_amt = (
            f"{AXIS_AMOUNT_COLUMN}_Axis"
            if f"{AXIS_AMOUNT_COLUMN}_Axis" in matched.columns
            else AXIS_AMOUNT_COLUMN
        )

        fin_amt = (
            f"{FINSTORE_AMOUNT_COLUMN}_Finstore"
            if f"{FINSTORE_AMOUNT_COLUMN}_Finstore" in matched.columns
            else FINSTORE_AMOUNT_COLUMN
        )

        matched["Variance"] = matched[axis_amt] - matched[fin_amt]
        # Get indices
        axis_idx = (
            "OriginalAxisIndex_Axis"
            if "OriginalAxisIndex_Axis" in matched.columns
            else "OriginalAxisIndex"
        )

        fin_idx = (
            "OriginalFinstoreIndex_Finstore"
            if "OriginalFinstoreIndex_Finstore" in matched.columns
            else "OriginalFinstoreIndex"
        )

        matched_axis_indices = set(matched[axis_idx].unique())
        matched_finstore_indices = set(matched[fin_idx].unique())
        remaining_axis = df_axis_pool[
            ~df_axis_pool["OriginalAxisIndex"].isin(matched_axis_indices)
        ].copy()

        remaining_finstore = df_finstore_pool[
            ~df_finstore_pool["OriginalFinstoreIndex"].isin(matched_finstore_indices)
        ].copy()

        unique_axis_matched = len(matched_axis_indices)
        
                # Drop temp columns
        cols_to_drop = [
            c for c in matched.columns
            if c.startswith("_match_key_")
        ]

        matched = matched.drop(columns=cols_to_drop, errors="ignore")
    print(
        f"  Priority {global_priority:2d} "
        f"({rule.category.value} #{rule.brd_priority}): "
        f"{unique_axis_matched:,} matches"
    )
    return matched, remaining_axis, remaining_finstore, {
        "global_priority": global_priority,
        "category": rule.category.value,
        "brd_priority": rule.brd_priority,
        "description": rule.description,
        "matched_rows": len(matched),
        "matched_unique_axis": unique_axis_matched
    }
print("Matching engine functions defined.")

✅ LAYER 1: BRD DETERMINISTIC MATCHING (15 Rules)


🔹 Section 8: Execute SOPHIS-Specific Matching (3 Rules)

In [ ]:
print(f"\n{'='*80}")
print("LAYER 1: BRD DETERMINISTIC MATCHING")
print(f"{'='*80}")
print(f"\nPHASE 1: SOPHIS-SPECIFIC MATCHING (3 Rules)")
print(f"{'-'*60}")

# Separate SOPHIS records
df_axis_sophis = df_axis[df_axis["SourceSystemName"].isin(SOPHIS_SYSTEMS)].copy()
df_axis_non_sophis = df_axis[~df_axis["SourceSystemName"].isin(SOPHIS_SYSTEMS)].copy()

print(f"SOPHIS Axis records: {len(df_axis_sophis):,}")
print(f"Non-SOPHIS Axis records: {len(df_axis_non_sophis):,}")
print("")

# Initialize tracking
all_matched_results = []
all_stats = []
global_priority = 0

# Execute SOPHIS rules
df_axis_sophis_remaining = df_axis_sophis.copy()
df_finstore_remaining = df_finstore.copy()

for rule in SOPHIS_MATCHING_RULES:
    global_priority += 1
    matched, df_axis_sophis_remaining, df_finstore_remaining, stats = execute_matching_rule(
        df_axis_sophis_remaining, df_finstore_remaining, rule, global_priority
    )

    if len(matched) > 0:
        all_matched_results.append(matched)
    all_stats.append(stats)

sophis_matched = sum(s.get("matched_unique_axis", 0) for s in all_stats)
print(f"\nSOPHIS Phase: {sophis_matched:,} / {len(df_axis_sophis):,} matched")

🔹 Section 9: Execute OTC Matching (10 Rules)

In [ ]:
print(f"\nPHASE 2: OTC MATCHING (10 Rules)")
print(f"{'-'*60}")

# Combine ALL remaining systems for OTC phase
df_axis_all_remaining = pd.concat(
    [
        df_axis_sophis_remaining,
        df_axis_non_sophis
    ],
    ignore_index=True
)

print(f"Records to process: {len(df_axis_all_remaining):,}")
print("")

df_axis_otc_remaining = df_axis_all_remaining.copy()

for rule in OTC_MATCHING_RULES:
    global_priority += 1
    matched, df_axis_otc_remaining, df_finstore_remaining, stats = execute_matching_rule(
        df_axis_otc_remaining, df_finstore_remaining, rule, global_priority
    )

    if len(matched) > 0:
        all_matched_results.append(matched)
    all_stats.append(stats)

otc_matched = sum(s.get("matched_unique_axis", 0) for s in all_stats if s["category"] == "OTC")
print(f"\nOTC Phase: {otc_matched:,} matches")

🔹 Section 10: Execute ETD Matching (2 Rules)

In [ ]:
print(f"\nPHASE 3: ETD MATCHING (2 Rules) - FALLBACK")
print(f"{'-'*60}")

# Get remaining ETD records
df_axis_etd_remaining = df_axis_otc_remaining[
    df_axis_otc_remaining["ReconSubProduct"] == "ETD"
].copy()

print(f"Remaining ETD records: {len(df_axis_etd_remaining):,}")
print("")

for rule in ETD_MATCHING_RULES:
    global_priority += 1
    matched, df_axis_etd_remaining, df_finstore_remaining, stats = execute_matching_rule(
        df_axis_etd_remaining, df_finstore_remaining, rule, global_priority
    )

    if len(matched) > 0:
        all_matched_results.append(matched)
    all_stats.append(stats)

etd_matched = sum(s.get("matched_unique_axis", 0) for s in all_stats if s["category"] == "ETD")
print(f"\nETD Phase: {etd_matched:,} matches")

🔹 Section 11: BRD Layer Results Summary

In [ ]:
print(f"\n{'='*80}")
print("LAYER 1 (BRD) RESULTS")
print(f"{'='*80}")

# Combine all BRD matches
if len(all_matched_results) > 0:
    df_brd_matched = pd.concat(all_matched_results, ignore_index=True)

    axis_idx_col = (
        "OriginalAxisIndex_Axis"
        if "OriginalAxisIndex_Axis" in df_brd_matched.columns
        else "OriginalAxisIndex"
    )

    brd_unique_axis_matched = df_brd_matched[axis_idx_col].nunique()
else:
    df_brd_matched = pd.DataFrame()
    brd_unique_axis_matched = 0


# Calculate unmatched after BRD
df_unmatched_axis_non_etd = df_axis_otc_remaining[
    df_axis_otc_remaining["ReconSubProduct"] != "ETD"
].copy()

df_brd_unmatched_axis = pd.concat(
    [
        df_unmatched_axis_non_etd,
        df_axis_etd_remaining
    ],
    ignore_index=True
)

brd_match_rate = brd_unique_axis_matched / ORIGINAL_AXIS_COUNT * 100

print("")
print(f"BRD Rules executed: 15")
print(f"Unique Axis matched: {brd_unique_axis_matched:,}")
print(f"BRD Match Rate: {brd_match_rate:.2f}%")
print(f"Remaining unmatched: {len(df_brd_unmatched_axis):,}")
print(f"Finstore remaining: {len(df_finstore_remaining):,}")
print(f"{'='*80}")

🔹 LAYER 2: GREEDY / PROBABILISTIC MATCHING


🔹 Section 12: Prepare Data for Greedy Matching

In [ ]:
print(f"\n{'='*80}")
print("LAYER 2: GREEDY/PROBABILISTIC MATCHING")
print(f"{'='*80}")

# Prepare unmatched data
df_axis_greedy = df_brd_unmatched_axis.copy()
df_finstore_greedy = df_finstore_remaining.copy()

# Clean numeric fields
df_axis_greedy["SACCRMTM"] = pd.to_numeric(
    df_axis_greedy["SACCRMTM"], errors="coerce"
)

df_axis_greedy["CounterpartyId_str"] = (
    df_axis_greedy["CounterpartyId"].astype(str).str.strip()
)

df_finstore_greedy["gbpequivalentamount"] = pd.to_numeric(
    df_finstore_greedy["gbpequivalentamount"], errors="coerce"
)

df_finstore_greedy["CounterpartyId_str"] = (
    df_finstore_greedy["CounterpartyId"].astype(str).str.strip()
)

print("")
print(f"Axis remaining for greedy matching: {len(df_axis_greedy):,}")
print(f"Finstore remaining: {len(df_finstore_greedy):,}")

# Initialize greedy tracking
greedy_matches = []
matched_axis_indices_greedy = set()
matched_finstore_indices_greedy = set()

🔹 Section 13: Greedy Strategy 1 – Amount + Counterparty (1%)

In [ ]:
print(
    f"\n--- Strategy 1: Amount + Counterparty "
    f"({GREEDY_TOLERANCE_PCT*100}% tolerance) ---"
)

strategy1_matches = []

# Group by counterparty for blocking
axis_by_cpty = df_axis_greedy.groupby("CounterpartyId_str")

for cpty, axis_group in axis_by_cpty:

    if cpty in ["nan", "", None]:
        continue

    # Find finstore records by counterparty
    fin_cpty = df_finstore_greedy[
        df_finstore_greedy["CounterpartyId_str"] == cpty
    ]

    if len(fin_cpty) == 0:
        continue

    for idx, axis_row in axis_group.iterrows():

        if axis_row["OriginalAxisIndex"] in matched_axis_indices_greedy:
            continue

        axis_amount = axis_row["SACCRMTM"]

        if pd.isna(axis_amount) or axis_amount == 0:
            continue

        tolerance = abs(axis_amount) * GREEDY_TOLERANCE_PCT

        candidates = fin_cpty[
            (~fin_cpty["OriginalFinstoreIndex"].isin(matched_finstore_indices_greedy)) &
            (abs(fin_cpty["gbpequivalentamount"] - axis_amount) <= tolerance)
        ]

        if len(candidates) > 0:
            candidates = candidates.copy()
            candidates["amount_diff"] = abs(
                candidates["gbpequivalentamount"] - axis_amount
            )

            best_match = candidates.nsmallest(1, "amount_diff").iloc[0]

            strategy1_matches.append({
                "OriginalAxisIndex_Axis": axis_row["OriginalAxisIndex"],
                "OriginalFinstoreIndex_Finstore": best_match["OriginalFinstoreIndex"],
                "SourceSystemName": axis_row["SourceSystemName"],
                "SACCRMTM_Axis": axis_amount,
                "gbpequivalentamount_Finstore": best_match["gbpequivalentamount"],
                "Variance": axis_amount - best_match["gbpequivalentamount"],
                "MatchGlobalPriority": 16,
                "MatchDescription": (
                    f"Greedy: Amount+Counterparty "
                    f"({GREEDY_TOLERANCE_PCT*100}%)"
                ),
                "MatchLayer": "GREEDY"
            })

            matched_axis_indices_greedy.add(axis_row["OriginalAxisIndex"])
            matched_finstore_indices_greedy.add(best_match["OriginalFinstoreIndex"])

print(f"Strategy 1 matches: {len(strategy1_matches):,}")

🔹 Section 14: Greedy Strategy 2 – Amount Only (0.1%)

In [ ]:
print(
    f"\n--- Strategy 2: Amount only "
    f"({STRICT_TOLERANCE_PCT*100}% tolerance) ---"
)

# Get remaining unmatched
axis_remaining_s2 = df_axis_greedy[
    ~df_axis_greedy["OriginalAxisIndex"].isin(matched_axis_indices_greedy)
].copy()

finstore_remaining_s2 = df_finstore_greedy[
    ~df_finstore_greedy["OriginalFinstoreIndex"].isin(matched_finstore_indices_greedy)
].copy()

print(f"Axis remaining after Strategy 1: {len(axis_remaining_s2):,}")

strategy2_matches = []

# Create amount buckets for blocking
def create_bucket(amount, size=1000):
    if pd.isna(amount):
        return None
    return int(amount / size) * size

axis_remaining_s2["amount_bucket"] = (
    axis_remaining_s2["SACCRMTM"].apply(create_bucket)
)

finstore_remaining_s2["amount_bucket"] = (
    finstore_remaining_s2["gbpequivalentamount"].apply(create_bucket)
)

# Group by bucket
axis_by_bucket = axis_remaining_s2.groupby("amount_bucket")

for bucket, axis_group in axis_by_bucket:

    if bucket is None:
        continue

    fin_bucket = finstore_remaining_s2[
        (finstore_remaining_s2["amount_bucket"] >= bucket - 1000) &
        (finstore_remaining_s2["amount_bucket"] <= bucket + 1000)
    ]

    if len(fin_bucket) == 0:
        continue

    for idx, axis_row in axis_group.iterrows():

        axis_amount = axis_row["SACCRMTM"]

        if pd.isna(axis_amount) or axis_amount == 0:
            continue

        tolerance = abs(axis_amount) * STRICT_TOLERANCE_PCT

        candidates = fin_bucket[
            (~fin_bucket["OriginalFinstoreIndex"].isin(matched_finstore_indices_greedy)) &
            (abs(fin_bucket["gbpequivalentamount"] - axis_amount) <= tolerance)
        ]

        if len(candidates) > 0:
            candidates = candidates.copy()
            candidates["amount_diff"] = abs(
                candidates["gbpequivalentamount"] - axis_amount
            )

            best_match = candidates.nsmallest(1, "amount_diff").iloc[0]

            strategy2_matches.append({
                "OriginalAxisIndex_Axis": axis_row["OriginalAxisIndex"],
                "OriginalFinstoreIndex_Finstore": best_match["OriginalFinstoreIndex"],
                "SourceSystemName": axis_row["SourceSystemName"],
                "SACCRMTM_Axis": axis_amount,
                "gbpequivalentamount_Finstore": best_match["gbpequivalentamount"],
                "Variance": axis_amount - best_match["gbpequivalentamount"],
                "MatchGlobalPriority": 17,
                "MatchDescription": (
                    f"Greedy: Amount Strict "
                    f"({STRICT_TOLERANCE_PCT*100}%)"
                ),
                "MatchLayer": "GREEDY"
            })

            matched_axis_indices_greedy.add(axis_row["OriginalAxisIndex"])
            matched_finstore_indices_greedy.add(best_match["OriginalFinstoreIndex"])

print(f"Strategy 2 matches: {len(strategy2_matches):,}")

🔹 Section 15: Greedy Layer Results

In [ ]:
print(f"\n{'='*60}")
print("LAYER 2 (GREEDY) SUMMARY")
print(f"{'='*60}")

if greedy_matches:
    df_greedy_matched = pd.DataFrame(greedy_matches)
    greedy_unique_axis = df_greedy_matched["OriginalAxisIndex_Axis"].nunique()
else:
    df_greedy_matched = pd.DataFrame()
    greedy_unique_axis = 0

print("")
print(f"Strategy 1 (Amount+Counterparty): {len(strategy1_matches):,}")
print(f"Strategy 2 (Amount Strict):       {len(strategy2_matches):,}")
print(f"Total Greedy matches:             {len(greedy_matches):,}")
print(f"Unique Axis matched (Greedy):     {greedy_unique_axis:,}")

🔹 Section 16: Final Results Consolidation

In [ ]:
print(f"\n{'='*80}")
print("FINAL RESULTS")
print(f"{'='*80}")

# Calculate totals
total_matched = brd_unique_axis_matched + greedy_unique_axis
final_match_rate = total_matched / ORIGINAL_AXIS_COUNT * 100

# Calculate final unmatched
all_matched_axis_ids = set()

if len(df_brd_matched) > 0:
    axis_col = (
        "OriginalAxisIndex_Axis"
        if "OriginalAxisIndex_Axis" in df_brd_matched.columns
        else "OriginalAxisIndex"
    )
    all_matched_axis_ids.update(df_brd_matched[axis_col].unique())

if len(df_greedy_matched) > 0:
    all_matched_axis_ids.update(df_greedy_matched["OriginalAxisIndex_Axis"].unique())

df_final_unmatched_axis = df_axis[
    ~df_axis["OriginalAxisIndex"].isin(all_matched_axis_ids)
]

print("")
print(f"SCOPE: {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}")
print(f"{'-'*60}")
print(f"Total Axis (in-scope): {ORIGINAL_AXIS_COUNT:,}")
print("")

print("Layer 1 (BRD Rules):")
print(f"  {brd_unique_axis_matched:,} matched ({brd_unique_axis_matched/ORIGINAL_AXIS_COUNT*100:.2f}%)")

print("Layer 2 (Greedy):")
print(f"  {greedy_unique_axis:,} matched ({greedy_unique_axis/ORIGINAL_AXIS_COUNT*100:.2f}%)")
print("")

print(f"TOTAL MATCHED:   {total_matched:,}")
print(f"TOTAL UNMATCHED: {len(df_final_unmatched_axis):,}")
print("")
print(f">>> FINAL MATCH RATE: {final_match_rate:.2f}% <<<")
print(f"{'='*80}")

🔹 Section 17: Greedy Matches by System

In [ ]:
print(f"\n{'='*60}")
print("GREEDY MATCHES BY SYSTEM")
print(f"{'='*60}")

if len(df_greedy_matched) > 0:
    system_breakdown = df_greedy_matched["SourceSystemName"].value_counts(ascending=False)
    for sys, count in system_breakdown.head(15).items():
        print(f"  {sys}: {count:,}")
else:
    print("  No greedy matches.")

🔹 Section 18: Remaining Unmatched by System

In [ ]:
print(f"\n{'='*60}")
print("REMAINING UNMATCHED BY SYSTEM")
print(f"{'='*60}")

unmatched_by_system = df_final_unmatched_axis["SourceSystemName"].value_counts().sort_values(ascending=False)
for sys, count in unmatched_by_system.head(15).items():
    print(f"  {sys}: {count:,}")

🔹 Section 19: Save Results

In [ ]:
print(f"Saving results to: {OUTPUT_DIR}")
print(f"{'-'*60}")

# Save BRD matches
if len(df_brd_matched) > 0:
    df_brd_matched.to_csv(OUTPUT_MATCHED_BRD, index=False)
    print(f"Saved: {OUTPUT_MATCHED_BRD.name} ({len(df_brd_matched):,} rows)")

# Save Greedy matches
if len(df_greedy_matched) > 0:
    df_greedy_matched.to_csv(OUTPUT_MATCHED_GREEDY, index=False)
    print(f"Saved: {OUTPUT_MATCHED_GREEDY.name} ({len(df_greedy_matched):,} rows)")

# Save combined matches
if len(df_brd_matched) > 0 or len(df_greedy_matched) > 0:
    frames_to_combine = []
    if len(df_brd_matched) > 0:
        frames_to_combine.append(df_brd_matched)
    if len(df_greedy_matched) > 0:
        frames_to_combine.append(df_greedy_matched)

    df_all_matched = pd.concat(frames_to_combine, ignore_index=True)
    df_all_matched.to_csv(OUTPUT_MATCHED_ALL, index=False)
    print(f"Saved: {OUTPUT_MATCHED_ALL.name} ({len(df_all_matched):,} rows)")

# Save unmatched
df_final_unmatched_axis.to_csv(OUTPUT_UNMATCHED_AXIS, index=False)
print(f"Saved: {OUTPUT_UNMATCHED_AXIS.name} ({len(df_final_unmatched_axis):,} rows)")

print("\nAll results saved successfully.")

🔹 Section 20: Summary Report


In [ ]:
# Generate summary report
report = f"""
{'='*80}
DERIVATIVES MATCHING - COMPLETE RESULTS (BRD + GREEDY)
{'='*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Scope: {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}

{'='*80}
INPUT DATA
{'='*80}
Total Axis (original): {ORIGINAL_AXIS_COUNT_FULL:,}
Total Finstore: {ORIGINAL_FINSTORE_COUNT:,}
Excluded (SOPHIS/DELTA1): {len(df_axis_excluded):,} trades if EXCLUDE_SOPHIS_DELTA1 else 0
Axis (in-scope): {ORIGINAL_AXIS_COUNT:,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING
{'='*80}
Rules executed: 15 (3 SOPHIS + 10 OTC + 2 ETD)
Unique Axis matched: {brd_unique_axis_matched:,}
Match Rate: {brd_match_rate:.2f}%

{'='*80}
LAYER 2: GREEDY/PROBABILISTIC MATCHING
{'='*80}
Strategy 1 (Amount+Counterparty, {GREEDY_TOLERANCE_PCT*100:.1f}%): {len(strategy1_matches):,}
Strategy 2 (Amount Strict, {STRICT_TOLERANCE_PCT*100:.1f}%): {len(strategy2_matches):,}
"""